# Week 1 — 데이터 탐색 + Classical Baseline

 한 세션에 데이터 처리 → sparse 문제 정의 → classical baseline까지.

## 이번 주 학습 목표
1. **데이터 I/O + 전처리** — `.bin` 로드, float 정규화, Otsu 임계값 자동 선택
2. **3D voxel 데이터 시각화** — 4개 도메인(BB·CastleGate·Bentheimer·Parker) 비교
3. **공극률 분석 + 등방성 검증** — slab별 분해, 세 축 프로파일
4. **슬라이스 보간 문제 정의** + **Classical baseline 정량 비교**
   - B1 (Linear) vs B2 (Cubic spline, scipy)
   - 평가 지표: `|Δφ|`(공극률 오차), `SSIM`(구조 유사도)
5. **이웃 거리 k sweep 으로 "왜 deep learning 이 필요한가" 정량 확인**

## 노트북 사용 방법

본 노트북의 코드는 모두 helpers/ 모듈에 정의되어 있고, 본문은 그 함수들을 호출해 결과를 확인하고 분석하는 흐름으로 구성됩니다. 파라미터를 다양하게 바꿔가며 결과 변화를 정량·정성적으로 검토합니다.

각 절에 본문과 함께 제시되는 심화 질문들은 본인의 추가 분석을 위한 출발점입니다. 모르는 용어는 helpers의 docstring을 참조.

마지막의 탐구 과제는 본 노트북을 본인 작업 사본으로 복사한 뒤, 코드 수정과 결과 분석을 함께 정리해 제출하는 형태입니다.

## 0. 환경 준비 + helper 함수 import

In [ ]:
import sys
from pathlib import Path
# helpers(dr_utils.py)는 같은 폴더(다운로드) 또는 ../helpers(저장소)에 있을 수 있음
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import numpy as np
import matplotlib.pyplot as plt

from dr_utils import (
    # 데이터 I/O + 전처리
    load_volume, porosity, normalize_to_float, otsu_threshold, binarize_otsu,
    # 시각화
    show_three_axis, porosity_profile,
    # 보간 baseline (이웃 거리 k)
    predict_linear_k, predict_cubic_k, neighbor_targets, eval_targets,
    linear_interpolate_slice,
    # 평가
    porosity_error, ssim_3d_mean,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
setup_plot_style()
print('환경 준비 완료')

## 1. 데이터 로드 — 4 도메인 동시 비교



| 도메인 | 출처 | 특징 |

|---|---|---|

| BB | Brazil sandstone | 기준 학습 도메인 |

| CastleGate | CastleGate sandstone | 고공극·불균질 |

| Bentheimer | Bentheimer sandstone | 균질 사암 |

| **Parker** | Parker sandstone | 저공극 (W1 신규 비교) |



모두 256³ uint8 binary (0=solid, 1=pore), voxel 2.25 μm.

In [ ]:
# data/ 는 같은 폴더(다운로드) 또는 ../data(저장소)에 있을 수 있음
DATA_DIR = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'BB_256.bin').exists()), Path('data'))

domains = {

    'BB':         load_volume(DATA_DIR / 'BB_256.bin'),

    'CastleGate': load_volume(DATA_DIR / 'CastleGate_256.bin'),

    'Bentheimer': load_volume(DATA_DIR / 'Bentheimer_256.bin'),

    'Parker':     load_volume(DATA_DIR / 'Parker_256.bin'),

}



print(f"{'Domain':<13} {'shape':<18} {'dtype':<8} {'φ (%)':>8}")

print('-' * 52)

for name, vol in domains.items():

    print(f"{name:<13} {str(vol.shape):<18} {str(vol.dtype):<8} {porosity(vol)*100:>7.2f}")

## 2. 데이터 전처리 — float 정규화 + Otsu 임계값



Deep learning 모델 학습 전 표준 전처리. W2 이후 UNet 입력 단계에서 매번 필요.



**`normalize_to_float`**: uint8 [0, 255] → float32 [0.0, 1.0]

**`otsu_threshold`**: grayscale에서 자동 임계값 (binary 데이터에선 0.5 — trivial)

In [ ]:
vol = domains['BB']

print(f'원본 dtype = {vol.dtype}, range = [{vol.min()}, {vol.max()}]')



vol_f = normalize_to_float(vol)

print(f'정규화 후 dtype = {vol_f.dtype}, range = [{vol_f.min()}, {vol_f.max()}]')



# Otsu — 본 binary 데이터에서는 trivial (값이 0 또는 1뿐)

t = otsu_threshold(vol_f[128])

print(f'Otsu threshold (z=128 slice) = {t:.4f}  (binary 데이터 → trivial)')

> **[직접 해보기]** Otsu는 grayscale 데이터에서 진가를 발휘. 인공 grayscale을 만들어 시각해봅시다.

In [ ]:
# 인공 grayscale: binary에 가우시안 노이즈 추가

rng = np.random.default_rng(0)

gray = vol[128].astype(np.float32) + rng.normal(0, 0.2, vol[128].shape)

gray = np.clip(gray, 0, 1)



t_gray = otsu_threshold(gray)

binary, _ = binarize_otsu(gray)



fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(gray); axes[0].set_title('인공 grayscale (noisy)')

axes[0].axis('off')

axes[1].hist(gray.ravel(), bins=80, color=GRAY)

axes[1].axvline(t_gray, color=ORANGE, lw=2, label=f'Otsu t={t_gray:.3f}')

axes[1].set_title('히스토그램 + Otsu'); axes[1].legend()

axes[2].imshow(binary); axes[2].set_title(f'Otsu binarize (φ={binary.mean()*100:.1f}%)')

axes[2].axis('off')

plt.tight_layout(); plt.show()

> **[직접 해보기]** Otsu가 자동으로 고른 임계값이 "공극과 암석의 경계"에 잘 맞나요?

> 만약 노이즈 크기(`rng.normal(0, 0.2, ...)` 의 `0.2`)를 0.05 또는 0.5로 바꾸면 Otsu 결과가 어떻게 변할까요? 직접 sweep해보세요.

## 3. 4 도메인 시각화 + 등방성 분석

In [ ]:
# 4 도메인 중앙 슬라이스 (z=128) 한 줄에

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (name, vol) in zip(axes, domains.items()):

    ax.imshow(vol[128])

    ax.set_title(f'{name}  (φ={porosity(vol)*100:.1f}%)')

    ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# BB의 세 축 프로파일 — 등방성 검증

vol = domains['BB']

n_slabs = 16

prof_z = porosity_profile(vol, axis=0, n_slabs=n_slabs)

prof_y = porosity_profile(vol, axis=1, n_slabs=n_slabs)

prof_x = porosity_profile(vol, axis=2, n_slabs=n_slabs)



fig, ax = plt.subplots(figsize=(9, 4))

xa = np.arange(n_slabs)

ax.plot(xa, prof_z, marker='o', label=f'z-axis (std={prof_z.std():.4f})', color=ORANGE, lw=2)

ax.plot(xa, prof_y, marker='s', label=f'y-axis (std={prof_y.std():.4f})', color=NAVY, lw=2)

ax.plot(xa, prof_x, marker='^', label=f'x-axis (std={prof_x.std():.4f})', color=GREEN, lw=2)

ax.axhline(porosity(vol), ls='--', color=GRAY, label=f'overall φ={porosity(vol):.3f}')

ax.set_xlabel(f'Slab index (n_slabs={n_slabs})'); ax.set_ylabel('φ')

ax.set_title('BB sandstone — 세 축 등방성 검증')

ax.legend(); plt.tight_layout(); plt.show()

> **[직접 해보기]** 4 도메인 각각에 대해 `prof_z.std()` 를 계산해 등방성 순위를 매겨보세요.

> 등방성이 좋은 도메인일수록 어느 축으로 sparse 측정을 하든 결과가 비슷할 것이라 기대할 수 있습니다 — 본인 실험으로 검증해보세요.

## 4. 문제 정의 — 슬라이스 보간

**상황**: micro-CT 는 슬라이스를 촘촘히 찍을수록 시간·비용이 큽니다. 듬성듬성 측정하면 아끼지만, 빠진 가운데 슬라이스들을 **이웃에서 예측(보간)**해야 합니다.

**이웃 거리 k**: 슬라이스 t 를 양옆 이웃 `t−k`, `t+k` 로 예측합니다. k 가 클수록 이웃이 멀어져(= 더 듬성듬성) **예측이 어려워집니다.**

**핵심 질문**: 이웃이 멀어질수록(k↑) 복원이 얼마나 나빠지나?

In [ ]:
vol = domains['BB']
t, k = 62, 3        # 슬라이스 t 를 이웃 거리 k 로 예측

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(vol[t - k]); axes[0].set_title(f'이웃 z={t-k}')
axes[1].imshow(vol[t]);     axes[1].set_title(f'예측 대상 z={t} (정답)', color=GREEN)
axes[2].imshow(vol[t + k]); axes[2].set_title(f'이웃 z={t+k}')
for ax in axes: ax.axis('off')
plt.suptitle(f'슬라이스 {t} 를 {t-k}·{t+k} 에서 예측 (이웃 거리 k={k})', y=1.03)
plt.tight_layout(); plt.show()

## 5. Classical Baseline — Linear (B1) vs Cubic (B2)

두 고전 보간법으로 각 슬라이스 t 를 이웃에서 복원하고 정량 비교.

- **B1 (Linear)**: 두 이웃 `t±k` 의 평균 — `0.5·(vol[t−k] + vol[t+k])`
- **B2 (Cubic)**: 4개 이웃 `t±k`, `t±3k` 를 쓰는 3차 spline (부드러움)

평가는 **예측한 슬라이스만** 대상으로 `|Δφ|`(공극률 오차, %p) 와 `SSIM`(구조 유사도, 0~1).

In [ ]:
vol = domains['BB']
k = 3

print(f'BB sandstone, 이웃 거리 k={k}')
print('-' * 50)
recon_l = predict_linear_k(vol, k)
m_l = eval_targets(recon_l, vol, k)
print(f'  B1 Linear   |Δφ|={m_l["dphi_pp"]:.2f}%p   SSIM={m_l["ssim"]:.4f}')
recon_c = predict_cubic_k(vol, k)
m_c = eval_targets(recon_c, vol, k)
print(f'  B2 Cubic    |Δφ|={m_c["dphi_pp"]:.2f}%p   SSIM={m_c["ssim"]:.4f}')

In [ ]:
# 시각: 원본 vs B1 vs B2 + 오차맵 (예측된 슬라이스 하나)
z_show = 62
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(vol[z_show]); axes[0].set_title(f'원본 z={z_show}')
axes[1].imshow(recon_l[z_show]); axes[1].set_title('B1 Linear')
axes[2].imshow(recon_c[z_show]); axes[2].set_title('B2 Cubic')
diff_l = np.abs(vol[z_show].astype(float) - recon_l[z_show])
axes[3].imshow(diff_l, cmap='hot'); axes[3].set_title('|원본 − B1|')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

> **[직접 해보기]** `k` 를 3, 5, 7, 10 으로 바꾸며 B1·B2 의 세 지표 변화를 표로 정리.

> 어느 k부터 cubic이 linear보다 의미 있게 좋은가요? (또는 그 반대?)

## 6. 4 도메인 × k sweep — 보간 난이도 곡선

이웃 거리 k 를 키우며(= 더 멀리서 예측) 복원 품질이 어떻게 나빠지는지 봅니다.
이 곡선이 "왜 deep learning이 필요한가" 의 정량 증거입니다. W2 에서 UNet 결과를 같은 plot 에 겹쳐 비교합니다.

In [ ]:
k_list = [1, 2, 3, 5, 7]
results = {name: {'k': [], 'dphi': [], 'ssim': []} for name in domains}

for name, vol in domains.items():
    for k in k_list:
        rec = predict_linear_k(vol, k)
        m = eval_targets(rec, vol, k)
        results[name]['k'].append(k)
        results[name]['dphi'].append(m['dphi_pp'])
        results[name]['ssim'].append(m['ssim'])
    print(f'  {name:<12}  done')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {'BB': ORANGE, 'CastleGate': NAVY, 'Bentheimer': GREEN, 'Parker': RED}
for name, r in results.items():
    axes[0].plot(r['k'], r['dphi'], marker='o', lw=2, label=name, color=colors[name])
    axes[1].plot(r['k'], r['ssim'], marker='s', lw=2, label=name, color=colors[name])
axes[0].set_xlabel('이웃 거리 k'); axes[0].set_ylabel('|Δφ| (%p)')
axes[0].set_title('Linear baseline — 공극률 오차 (k↑ → 나빠짐)')
axes[1].set_xlabel('이웃 거리 k'); axes[1].set_ylabel('SSIM')
axes[1].set_title('Linear baseline — 구조 유사도 (k↑ → 나빠짐)')
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **[직접 해보기]** 4 도메인 중 어느 사암이 "sparse 보간이 가장 어려운가"?

> 그 이유로 어떤 가설을 세울 수 있을까요? (힌트: 공극률, 등방성, 구조 복잡도)

## 7. 다음 주 (W2)

- 본 W1 baseline 곡선을 "deep learning 이 어디까지 끌어내릴 수 있나" 직접 확인
- mini UNet (~30K–120K params) 을 학생 노트북(CPU)에서 직접 학습
- `pip install torch torchvision`

W1 에서 정량적으로 확인한 baseline 의 한계 — 어디서 가장 답답했나요? 그 지점이 W2 에서 deep learning 을 만나는 출발점입니다.

---

## 🎯 W1 탐구 과제

다음 과제는 본 노트북의 코드를 수정·확장하며 결과 분석과 함께 정리합니다.

### 과제 1 — 전수 baseline 비교 (필수)
4 도메인 × {B1 Linear, B2 Cubic} × 이웃 거리 k ∈ {1, 2, 3, 5, 7} 의 모든 조합에 대해 (|Δφ|, SSIM) 을 측정 → 표(예: pandas DataFrame)로 정리·해석.
- 모든 도메인에서 B2(Cubic)가 B1(Linear)보다 나은가? 반례 도메인의 특징은?
- 이웃 거리 k 가 커질수록 어느 baseline 이 더 빨리 나빠지나?

### 과제 2 — 도메인별 난이도 (필수)
4 도메인의 k-sweep 곡선(|Δφ|·SSIM)을 겹쳐, "어느 사암이 보간이 가장 어려운가" 를 정량 비교하고, 그 도메인의 구조적 특징(공극률·불균질성 등)으로 이유를 설명.

### 과제 3 — 이진화 임계값 (선택)
`predict_linear_k` 의 이진화 임계값(`> 0.5`)을 바꾸거나, 인공 grayscale 에 noise σ 를 다양하게 주며 `binarize_otsu` 가 어디서 불안정해지는지 실험·시각화.

### 과제 4 — per-slice 오차 분석 (선택, 도전)
복원 오차가 모든 슬라이스에서 균일한가? 슬라이스별 |Δφ| 를 그래프로 그리고, 오차가 큰 슬라이스들의 구조적 공통점을 찾아보세요.